In [ ]:
import pandas as pd
import torch
from transformers import EncoderDecoderModel, AutoTokenizer
from tqdm import tqdm
import numpy as np
from difflib import SequenceMatcher
import os

# 평가 metrics

In [ ]:
def calculate_char_accuracy(pred, target):
    """문자 단위 정확도"""
    if len(target) == 0:
        return 0.0
    correct = sum(1 for p, t in zip(pred, target) if p == t)
    return correct / max(len(pred), len(target))

def calculate_word_accuracy(pred, target):
    """단어 단위 정확도"""
    pred_words = pred.split()
    target_words = target.split()
    if len(target_words) == 0:
        return 0.0
    correct = sum(1 for p, t in zip(pred_words, target_words) if p == t)
    return correct / max(len(pred_words), len(target_words))

def calculate_similarity(pred, target):
    """문자열 유사도 (SequenceMatcher)"""
    return SequenceMatcher(None, pred, target).ratio()

# data load

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
print("데이터 로딩 중...")
train = pd.read_csv('/content/drive/MyDrive/data/open/train.csv', encoding = 'utf-8-sig')
test = pd.read_csv('/content/drive/MyDrive/data/open/test.csv', encoding = 'utf-8-sig')

In [ ]:
# 평가용 샘플 (전체 사용 시 시간이 오래 걸릴 수 있음)
train_samples = train[:200]  # Few-shot 예시용
eval_samples = train[200:300]  # 평가용 (train의 일부를 평가용으로)

In [ ]:
# Few-shot 예시 생성
samples = []
for i in range(10):
    sample = f"input : {train_samples['input'].iloc[i]} \n output : {train_samples['output'].iloc[i]}"
    samples.append(sample)

In [ ]:
# 평가할 모델 리스트
encoder_models = [
    "kakaobank/kf-deberta-base",
    "monologg/koelectra-small-v2-discriminator",
    "klue/roberta-small",
    "beomi/KcELECTRA-base",
]

In [ ]:
# 결과 파일 경로
results_file_path = '/content/drive/MyDrive/data/open/model_evaluation_results.csv'

# 기존 결과 파일 로드
already_evaluated = []
if os.path.exists(results_file_path):
    print("\n기존 평가 결과 파일을 찾았습니다.")
    existing_results = pd.read_csv(results_file_path, encoding='utf-8-sig')
    already_evaluated = existing_results['model_name'].tolist()
    print(f"이미 평가된 모델: {already_evaluated}")

def save_result_immediately(result_dict, results_file_path):
    """각 모델 평가 후 즉시 결과를 파일에 저장"""
    if os.path.exists(results_file_path):
        existing = pd.read_csv(results_file_path, encoding='utf-8-sig')
        existing = existing[existing['model_name'] != result_dict['model_name']]
        updated = pd.concat([existing, pd.DataFrame([result_dict])], ignore_index=True)
    else:
        updated = pd.DataFrame([result_dict])

    updated = updated.sort_values('avg_score', ascending=False).reset_index(drop=True)
    updated.to_csv(results_file_path, index=False, encoding='utf-8-sig')
    print(f"✓ 결과가 저장되었습니다: {results_file_path}")

In [ ]:
results = []

for encoder_model_name in encoder_models:
    # EncoderDecoder 모델명 생성
    model_name = f"{encoder_model_name}-encoder-decoder"

    if model_name in already_evaluated:
        print(f"\n{'='*60}")
        print(f"모델 '{model_name}'은 이미 평가되었습니다. 건너뜁니다.")
        print(f"{'='*60}\n")
        continue

    print(f"\n{'='*60}")
    print(f"모델 평가 중: {model_name}")
    print(f"인코더: {encoder_model_name}")
    print(f"{'='*60}\n")

    try:
        # 토크나이저 로드
        tokenizer = AutoTokenizer.from_pretrained(encoder_model_name)

        # pad_token 설정
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token or tokenizer.unk_token or '[PAD]'
            tokenizer.add_special_tokens({'pad_token': '[PAD]'})

        if tokenizer.bos_token is None:
            if tokenizer.cls_token:
                tokenizer.bos_token = tokenizer.cls_token
            else:
                tokenizer.add_special_tokens({'bos_token': '[BOS]'})

        print(f"특수 토큰 설정:")
        print(f"  - pad_token: {tokenizer.pad_token} (id: {tokenizer.pad_token_id})")
        print(f"  - bos_token: {tokenizer.bos_token} (id: {tokenizer.bos_token_id})")
        print(f"  - eos_token: {tokenizer.eos_token} (id: {tokenizer.eos_token_id})")

        print("EncoderDecoder 모델 생성 중...")
        # EncoderDecoder 모델 생성 (같은 모델을 encoder와 decoder로 사용)
        model = EncoderDecoderModel.from_encoder_decoder_pretrained(
            encoder_model_name,
            encoder_model_name,
            tie_encoder_decoder=True  # 파라미터 공유로 메모리 절약
        )

        # 토크나이저 크기 조정 (새로운 특수 토큰 추가된 경우)
        model.encoder.resize_token_embeddings(len(tokenizer))
        model.decoder.resize_token_embeddings(len(tokenizer))

        # 특수 토큰 설정 (필수!)
        model.config.decoder_start_token_id = tokenizer.bos_token_id
        model.config.bos_token_id = tokenizer.bos_token_id
        model.config.eos_token_id = tokenizer.eos_token_id
        model.config.pad_token_id = tokenizer.pad_token_id

        # 추가 설정
        model.config.vocab_size = model.config.encoder.vocab_size
        model.config.max_length = 128
        model.config.min_length = 5
        model.config.no_repeat_ngram_size = 3

        print(f"모델 설정:")
        print(f"  - decoder_start_token_id: {model.config.decoder_start_token_id}")
        print(f"  - bos_token_id: {model.config.bos_token_id}")
        print(f"  - eos_token_id: {model.config.eos_token_id}")
        print(f"  - pad_token_id: {model.config.pad_token_id}")

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model.to(device)

        # Fine-tuning (간단한 학습)
        print("모델 Fine-tuning 중... (간단한 학습)")
        model.train()
        optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

        # 일부 데이터로만 빠르게 학습
        num_train_samples = min(100, len(train_samples))  # 100개만 사용
        for epoch in range(1):  # 1 epoch만
            for idx in tqdm(range(num_train_samples), desc=f"Training"):
                input_text = train_samples['input'].iloc[idx]
                target_text = train_samples['output'].iloc[idx]

                # 입력과 타겟 인코딩
                inputs = tokenizer(
                    input_text,
                    return_tensors="pt",
                    max_length=128,
                    truncation=True,
                    padding="max_length"
                ).to(device)

                labels = tokenizer(
                    target_text,
                    return_tensors="pt",
                    max_length=128,
                    truncation=True,
                    padding="max_length"
                ).input_ids.to(device)

                # -100으로 패딩 마스킹
                labels[labels == tokenizer.pad_token_id] = -100

                # Forward pass
                outputs = model(
                    input_ids=inputs.input_ids,
                    attention_mask=inputs.attention_mask,
                    labels=labels
                )

                loss = outputs.loss
                loss.backward()
                optimizer.step()
                optimizer.zero_grad()

        print("Fine-tuning 완료! 평가 시작...")
        model.eval()

        # 평가 메트릭 저장
        char_accuracies = []
        word_accuracies = []
        similarities = []

        # 각 샘플에 대해 추론
        for idx, row in tqdm(eval_samples.iterrows(), total=len(eval_samples), desc=f"Evaluating {model_name}"):
            query = row['input']
            target = row['output']

            try:
                inputs = tokenizer(
                    query,
                    return_tensors="pt",
                    max_length=128,
                    truncation=True
                ).to(device)

                with torch.no_grad():
                    outputs = model.generate(
                        **inputs,
                        max_length=128,
                        num_beams=4,
                        early_stopping=True,
                        decoder_start_token_id=model.config.decoder_start_token_id,
                        bos_token_id=model.config.bos_token_id,
                        eos_token_id=model.config.eos_token_id,
                        pad_token_id=model.config.pad_token_id
                    )

                result = tokenizer.decode(outputs[0], skip_special_tokens=True)

                # 메트릭 계산
                char_acc = calculate_char_accuracy(result, target)
                word_acc = calculate_word_accuracy(result, target)
                sim = calculate_similarity(result, target)

                char_accuracies.append(char_acc)
                word_accuracies.append(word_acc)
                similarities.append(sim)

            except Exception as e:
                print(f"Error processing sample {idx}: {str(e)}")
                continue

        # 평균 메트릭 계산
        avg_char_acc = np.mean(char_accuracies) if char_accuracies else 0.0
        avg_word_acc = np.mean(word_accuracies) if word_accuracies else 0.0
        avg_sim = np.mean(similarities) if similarities else 0.0

        # 결과 저장
        result_dict = {
            "model_name": model_name,
            "char_accuracy": avg_char_acc,
            "word_accuracy": avg_word_acc,
            "similarity": avg_sim,
            "avg_score": (avg_char_acc + avg_word_acc + avg_sim) / 3,
            "model_type": "EncoderDecoder",
            "encoder": encoder_model_name
        }
        results.append(result_dict)

        # 즉시 파일에 저장
        save_result_immediately(result_dict, results_file_path)

        print(f"\n[{model_name}] 평가 결과:")
        print(f"  - 문자 정확도: {avg_char_acc:.4f}")
        print(f"  - 단어 정확도: {avg_word_acc:.4f}")
        print(f"  - 유사도: {avg_sim:.4f}")
        print(f"  - 평균 점수: {result_dict['avg_score']:.4f}")

        # 메모리 및 디스크 정리
        del model
        del tokenizer
        del optimizer
        torch.cuda.empty_cache()

    except Exception as e:
        print(f"Error loading model {encoder_model_name}: {str(e)}")
        import traceback
        traceback.print_exc()

        error_result = {
            "model_name": model_name,
            "char_accuracy": 0.0,
            "word_accuracy": 0.0,
            "similarity": 0.0,
            "avg_score": 0.0,
            "model_type": "EncoderDecoder",
            "encoder": encoder_model_name,
            "error": str(e)
        }
        results.append(error_result)
        save_result_immediately(error_result, results_file_path)
        continue

In [ ]:
# 결과 파일 경로
results_file_path = '/content/drive/MyDrive/data/open/model_evaluation_results.csv'

# 기존 결과 파일이 있으면 로드
import os
if os.path.exists(results_file_path):
    print("\n기존 평가 결과 파일을 찾았습니다. 결과를 추가합니다...")
    existing_results = pd.read_csv(results_file_path, encoding='utf-8-sig')

    # 새로운 결과를 데이터프레임으로 변환
    new_results_df = pd.DataFrame(results)

    # 기존 결과와 병합 (중복 모델은 새 결과로 업데이트)
    combined_results = pd.concat([existing_results, new_results_df], ignore_index=True)
    combined_results = combined_results.drop_duplicates(subset=['model_name'], keep='last')
    results_df = combined_results.sort_values('avg_score', ascending=False).reset_index(drop=True)
else:
    print("\n새로운 평가 결과 파일을 생성합니다...")
    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values('avg_score', ascending=False).reset_index(drop=True)

In [ ]:
print("\n" + "="*60)
print("최종 평가 결과 (성능 순)")
print("="*60)
print(results_df.to_string(index=False))

In [ ]:
# 결과 저장
results_df.to_csv(results_file_path, index=False, encoding='utf-8-sig')
print(f"\n평가 결과가 '{results_file_path}'에 저장되었습니다.")
print(f"총 {len(results_df)}개의 모델이 평가되었습니다.")